In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy.stats import shapiro, friedmanchisquare, wilcoxon

In [33]:
SCORE_CONDITIONS = ["standard", "voice_control", "plugin"]

sus_rtlx_df = pd.read_csv("sus_rtlx_long.csv")

sus_wide = sus_rtlx_df.pivot(index="participant", columns="condition", values="SUS")[SCORE_CONDITIONS]
rtlx_wide = sus_rtlx_df.pivot(index="participant", columns="condition", values="RTLX")[SCORE_CONDITIONS]

In [34]:
sus_wide.head()

condition,standard,voice_control,plugin
participant,,,
1,100.0,10.0,47.5
2,97.5,47.5,60.0
3,97.5,15.0,47.5
4,80.0,62.5,85.0
5,97.5,57.5,45.0


In [35]:
rtlx_wide.head()

condition,standard,voice_control,plugin
participant,,,
1,0.000000,67.500000,60.000000
2,8.333333,45.000000,29.166667
3,7.500000,78.333333,44.166667
4,6.666667,51.666667,20.000000
5,0.000000,50.833333,47.500000


# Shapiro-Wilk Normality Test

In [36]:
def print_shapiro_result(condition_name, scores):
    stat, p = shapiro(scores)
    if p > 0.05:
        verdict = "normal"
    else:
        verdict = "NOT normal"
    print(f"  {condition_name}: W = {stat:.3f}, p = {p:.4f} -> {verdict}")

print("Shapiro-Wilk normality test")

print("\nSUS")
print_shapiro_result("standard", sus_wide["standard"])
print_shapiro_result("voice_control", sus_wide["voice_control"])
print_shapiro_result("plugin", sus_wide["plugin"])

print("\nRTLX")
print_shapiro_result("standard", rtlx_wide["standard"])
print_shapiro_result("voice_control", rtlx_wide["voice_control"])
print_shapiro_result("plugin", rtlx_wide["plugin"])

Shapiro-Wilk normality test

SUS
  standard: W = 0.876, p = 0.0416 -> NOT normal
  voice_control: W = 0.926, p = 0.2344 -> normal
  plugin: W = 0.900, p = 0.0948 -> normal

RTLX
  standard: W = 0.877, p = 0.0426 -> NOT normal
  voice_control: W = 0.961, p = 0.7154 -> normal
  plugin: W = 0.968, p = 0.8197 -> normal


# Friedman Test

In [37]:
# Friedman omnibus test for SUS
sus_standard = sus_wide["standard"]
sus_voice_control = sus_wide["voice_control"]
sus_plugin = sus_wide["plugin"]

sus_stat, sus_p = friedmanchisquare(sus_standard, sus_voice_control, sus_plugin)

print(f"SUS:  chi2 = {sus_stat:.3f},  p = {sus_p:.7f}")

# Friedman omnibus test for RTLX
rtlx_standard = rtlx_wide["standard"]
rtlx_voice_control = rtlx_wide["voice_control"]
rtlx_plugin = rtlx_wide["plugin"]

rtlx_stat, rtlx_p = friedmanchisquare(rtlx_standard, rtlx_voice_control, rtlx_plugin)

print(f"RTLX: chi2 = {rtlx_stat:.3f}, p = {rtlx_p:.7f}")

SUS:  chi2 = 22.933,  p = 0.0000105
RTLX: chi2 = 27.138, p = 0.0000013


# Wilcoxon Test

In [38]:
from scipy.stats import wilcoxon
from itertools import combinations

# three pairwise comparisons per measure, Bonferroni-corrected
BONFERRONI_ALPHA = 0.05 / 3  # 3 comparisons -> 0.0167

def print_wilcoxon_result(wide_df, cond_a, cond_b):
    result = wilcoxon(wide_df[cond_a], wide_df[cond_b])
    stat = float(result.statistic)
    p = float(result.pvalue)

    if p < BONFERRONI_ALPHA:
        verdict = "significant"
    else:
        verdict = "not significant"

    print(f"  {cond_a} vs {cond_b}: W = {stat:.3f}, p = {p:.4f} -> {verdict}")

print(f"Pairwise Wilcoxon signed-rank tests (Bonferroni alpha = {BONFERRONI_ALPHA:.4f})")

print("\nSUS")
print_wilcoxon_result(sus_wide, "standard", "voice_control")
print_wilcoxon_result(sus_wide, "standard", "plugin")
print_wilcoxon_result(sus_wide, "voice_control", "plugin")

print("\nRTLX")
print_wilcoxon_result(rtlx_wide, "standard", "voice_control")
print_wilcoxon_result(rtlx_wide, "standard", "plugin")
print_wilcoxon_result(rtlx_wide, "voice_control", "plugin")

Pairwise Wilcoxon signed-rank tests (Bonferroni alpha = 0.0167)

SUS
  standard vs voice_control: W = 0.000, p = 0.0006 -> significant
  standard vs plugin: W = 1.000, p = 0.0008 -> significant
  voice_control vs plugin: W = 11.000, p = 0.0053 -> significant

RTLX
  standard vs voice_control: W = 0.000, p = 0.0007 -> significant
  standard vs plugin: W = 0.000, p = 0.0010 -> significant
  voice_control vs plugin: W = 4.500, p = 0.0026 -> significant
